# 07 · Model assurance and monitoring

### Session 3 take-home | Safiri Telecom churn case

---

Notebook 05 ended with a model card and a list of retraining triggers. This notebook
is what you do **before** you sign that card and **after** the model goes live.

Three questions, none of which you can answer from an AUC:

1. **What does this model actually weigh?** You have never read your own coefficients.
2. **What does it do to people?** Somebody will ask. You either tested or you did not.
3. **How will you know when it stops being the model you approved?** Input monitoring
   is not enough, and this notebook shows you exactly why.

### What is already covered elsewhere

This notebook does not re-teach. Where something is established, it is cited:

| Already yours | Where |
|---|---|
| The six questions, cost of error in currency | NB00 Parts 1 and 10 |
| Leakage, temporal splits, the non-ML baseline | NB00 Parts 4-8 |
| Precision at k, net campaign value, capacity | NB00 Part 9, NB04 §1-2 |
| Pipelines, cross-validation, selecting on value | NB04 §4-6 |
| Segment error analysis | NB04 §8 |
| Serving contracts, skew, PSI, drift, the model card | NB05 §1-5 |

### Cell tags

| Tag | What you do |
|---|---|
| **READ** | Run it, read the output, answer the question. No typing. |
| **FILL** | Change one clearly marked value. |
| **BUILD** | Open-ended. Write your own code. |

## 0. Setup

If you finished a model in Session 2, point `MY_SCORES` at it. If you did not, leave
everything as it is: the notebook rebuilds the reference model from scratch in about
fifteen seconds and every cell below works identically.

In [ ]:
import numpy as np
import pandas as pd

def make_panel(seed=42, n_subscribers=9000, n_months=8):
    """Simulate a monthly prepaid subscriber panel for Safiri Telecom."""
    rng = np.random.default_rng(seed)
    n = n_subscribers
    region  = rng.choice(["Nairobi","Coast","Western","Rift Valley","Eastern"], n, p=[.34,.16,.18,.20,.12])
    plan    = rng.choice(["Prepaid Basic","Prepaid Plus","Data Bundle"], n, p=[.55,.28,.17])
    handset = rng.choice(["Feature","Smart-Entry","Smart-Mid","Smart-High"], n, p=[.22,.38,.28,.12])
    tenure0, base_arpu = rng.integers(1,72,n), rng.lognormal(5.6,.55,n)
    heavy_data, eps = rng.random(n) < .30, rng.normal(0,.55,n)

    r1 = (-1.0 - 0.032*np.clip(tenure0,0,48) - 0.0014*base_arpu
          + 0.60*(plan=="Prepaid Basic") + 0.50*(handset=="Feature")
          + 0.30*(region=="Western") + eps)
    r2 = (-0.55 - 0.006*np.clip(tenure0,0,48) + 0.0003*base_arpu + 1.15*heavy_data
          + 0.70*np.isin(handset,["Smart-Mid","Smart-High"]) + 0.20*(plan=="Prepaid Basic") + eps)
    p1, p2 = 1/(1+np.exp(-r1)), 1/(1+np.exp(-r2))

    churn_month = np.full(n, 99)
    for m in range(1, n_months+1):
        hz = np.clip((p1 if m <= 5 else p2)*0.20, 0, .9)
        churn_month[(rng.random(n) < hz) & (churn_month == 99)] = m

    rows = []
    for m in range(1, n_months+1):
        alive = churn_month >= m
        decay = np.where(np.clip(churn_month-m,0,6) <= 2, 0.66 + 0.17*np.clip(churn_month-m,0,6), 1.0)
        rows.append(pd.DataFrame({
            "subscriber_id": np.arange(n), "month": m, "region": region, "plan": plan,
            "handset_class": handset, "tenure_months": tenure0 + m,
            "topup_kes": np.where(alive, base_arpu*decay*rng.lognormal(0,.30,n), 0).round(0),
            "topup_count": np.where(alive, rng.poisson(np.clip(6*decay,0.3,None)), 0),
            "data_mb": np.where(alive, base_arpu*decay*rng.lognormal(0,.5,n)*(1+1.4*heavy_data), 0).round(1),
            "voice_minutes": np.where(alive, rng.gamma(3, 22*decay), 0).round(1),
            "days_since_last_topup": np.where(alive,
                np.clip(rng.poisson(np.clip(20*(1-decay)+4,1,None)),0,60), 60),
            "complaints_90d": rng.poisson(0.18 + 0.45*((churn_month-m) <= 2), n),
            "active": alive.astype(int), "_churn_month": churn_month}))
    return pd.concat(rows, ignore_index=True)

def build_snapshot(panel, prediction_month, horizon=2):
    """Features as at the prediction point; label from the outcome window."""
    obs = panel[(panel.month == prediction_month) & (panel.active == 1)].copy()
    cm = obs._churn_month.values
    obs["churned"] = ((cm > prediction_month) & (cm <= prediction_month + horizon)).astype(int)
    return obs.drop(columns=["_churn_month", "active", "month"])

FEATURES_NUM = ["tenure_months","topup_kes","topup_count","data_mb",
                "voice_minutes","days_since_last_topup","complaints_90d"]
FEATURES_CAT = ["region","plan","handset_class"]

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import cross_val_score
from sklearn.metrics import roc_auc_score

sns.set_style("whitegrid")
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 170)
plt.rcParams["figure.figsize"] = (10, 4.5)

# ---- the economics, unchanged from NB00 Part 1 -------------------------------
CALL_CAPACITY, OFFER_COST_KES, MARGIN_KES, TAKE_UP = 500, 200, 2400, 0.35
BREAKEVEN = OFFER_COST_KES / (MARGIN_KES * TAKE_UP)

def enrich(d, fit=False, T={}):
    d = d.copy()
    if fit:
        T["heavy_data"] = d.data_mb.quantile(.75)
        T["low_topup"]  = d.topup_kes.quantile(.25)
    d["heavy_data_user"]   = (d.data_mb   > T["heavy_data"]).astype(int)
    d["low_value"]         = (d.topup_kes < T["low_topup"]).astype(int)
    d["has_complained"]    = (d.complaints_90d >= 1).astype(int)
    d["lapsed"]            = (d.days_since_last_topup > 21).astype(int)
    d["revenue_per_topup"] = d.topup_kes / d.topup_count.clip(lower=1)
    d["smartphone"]        = d.handset_class.isin(["Smart-Mid","Smart-High"]).astype(int)
    return d

THRESH = {}
panel = make_panel()
train = enrich(pd.concat([build_snapshot(panel,3), build_snapshot(panel,4)], ignore_index=True), True, THRESH)
test  = enrich(build_snapshot(panel, 6), False, THRESH)

NUM  = FEATURES_NUM + ["revenue_per_topup"]
FLAG = ["heavy_data_user","low_value","has_complained","lapsed","smartphone"]
CAT  = FEATURES_CAT

def preprocessor(num=None, cat=None):
    return ColumnTransformer([("num", StandardScaler(), num or NUM+FLAG),
                              ("cat", OneHotEncoder(handle_unknown="ignore"), cat or CAT)])

X_tr, y_tr = train[NUM+FLAG+CAT], train.churned
X_te, y_te = test[NUM+FLAG+CAT],  test.churned

logit = Pipeline([("pre", preprocessor()),
                  ("clf", LogisticRegression(max_iter=3000, class_weight="balanced", C=5))]).fit(X_tr, y_tr)
booster = Pipeline([("pre", preprocessor()),
                    ("clf", GradientBoostingClassifier(random_state=123, n_estimators=150,
                                                       learning_rate=.05, max_depth=2))]).fit(X_tr, y_tr)

score_logit = logit.predict_proba(X_te)[:, 1]
score_gb    = booster.predict_proba(X_te)[:, 1]
score_rule  = test.complaints_90d.values          # the incumbent, from NB00 Part 8

# FILL: if you have your own Session 2 scores for this test set, assign them here.
MY_SCORES = None
if MY_SCORES is not None:
    score_gb = np.asarray(MY_SCORES)

print(f"train {train.shape}  churn {train.churned.mean():.1%}")
print(f"test  {test.shape}  churn {test.churned.mean():.1%}")
print(f"break-even precision: {BREAKEVEN:.1%}")

In [ ]:
# Helpers from NB00 Part 9 and NB04 §1. Nothing new here.
def precision_at_k(scores, truth=None, k=CALL_CAPACITY):
    truth = y_te.values if truth is None else truth
    idx = np.argsort(-np.asarray(scores))[:k]
    hits = np.asarray(truth)[idx]
    return hits.mean(), hits.sum()

def campaign_value(scores, truth=None, k=CALL_CAPACITY):
    _, hits = precision_at_k(scores, truth, k)
    return hits * TAKE_UP * MARGIN_KES - k * OFFER_COST_KES

print(f"{'instrument':<22}{'AUC':>7}{'precision@500':>16}{'net value KES':>16}")
print("-"*61)
for name, s in [("logistic regression", score_logit),
                ("gradient boosting",   score_gb),
                ("complaints rule",     score_rule)]:
    p,_ = precision_at_k(s)
    print(f"{name:<22}{roc_auc_score(y_te, s):>7.3f}{p:>15.1%}{campaign_value(s):>16,.0f}")

**READ and answer.** Hold on to this table. Section 6 comes back to it.

> *The complaints rule has the worst AUC of the three and beats one of the models on
> money. Which one, and by how much?*

---

# 1. Your coefficients are an audit surface

A logistic regression is a list of signed weights. One line of code turns your model
into a table that a regulator, a risk officer or an affected subscriber can argue with.

In [ ]:
feature_names = [f.split("__", 1)[1] for f in logit.named_steps["pre"].get_feature_names_out()]
coefficients  = logit.named_steps["clf"].coef_[0]

audit = pd.DataFrame({"feature": feature_names, "coefficient": coefficients})
audit["direction"] = np.where(audit.coefficient > 0, "raises churn risk", "lowers churn risk")
audit["magnitude"] = audit.coefficient.abs()

print(audit.sort_values("magnitude", ascending=False).head(12)
           .drop(columns="magnitude").round(3).to_string(index=False))

**READ and answer.** Go down the list and sort each row into one of two piles.

The behavioural features are easy. Complaints raise risk, a lapsed top-up raises risk,
long tenure lowers it. You can say any of those out loud without flinching.

Then there are the rows naming a region, a plan or a handset band.

> *Pick the largest of those. Write the sentence you would say to a subscriber who
> asked why they were, or were not, contacted.*

Nobody decided to target on handset class. The model found weight there because the
training data had it. **An audit is not a search for intent. It is a search for effect.**

---

# 2. The proxy trap

The obvious response to section 1 is to remove the offending column. Before you do,
test whether the rest of the data already carries it.

The test is three lines: try to predict the attribute from the behavioural features
alone. If you can, the attribute is encoded, and removing the column removes your
ability to see it rather than the model's ability to use it.

In [ ]:
behavioural = NUM + FLAG

print(f"{'attribute':<18}{'majority baseline':>19}{'predicted from behaviour':>27}")
print("-"*64)
proxy_results = {}
for attribute in ["handset_class", "region", "plan"]:
    baseline = train[attribute].value_counts(normalize=True).max()
    accuracy = cross_val_score(
        RandomForestClassifier(n_estimators=120, min_samples_leaf=20, random_state=0, n_jobs=-1),
        train[behavioural], train[attribute], cv=3, scoring="accuracy").mean()
    proxy_results[attribute] = (baseline, accuracy)
    print(f"{attribute:<18}{baseline:>18.1%}{accuracy:>26.1%}")

**READ and answer.** One of those three is strongly encoded. The other two are not.

> *Which one, and what real-world quantity is it standing in for?*

<br>

---

Handset class is predictable from data use and top-up behaviour well above its
baseline. It is a proxy for income. **The model targets on income whether or not you
give it the column.**

Region came out clean, and you should know why: this data is simulated, and the
generator assigned region independently of everything else. In a real Kenyan telco
extract it would not be independent, and you would find out by running exactly this
test rather than by assuming either way.

That is the transferable part. Not the result, the test.

## 2.1 So remove the column and see what changes

In [ ]:
# Train a second model that never sees handset class in any form.
NUM_BLIND = [c for c in NUM+FLAG if c != "smartphone"]
CAT_BLIND = ["region", "plan"]

blind = Pipeline([("pre", preprocessor(NUM_BLIND, CAT_BLIND)),
                  ("clf", LogisticRegression(max_iter=3000, class_weight="balanced", C=5))]
                ).fit(train[NUM_BLIND+CAT_BLIND], y_tr)
score_blind = blind.predict_proba(test[NUM_BLIND+CAT_BLIND])[:, 1]

def called_mix(scores, column="handset_class"):
    idx = np.argsort(-np.asarray(scores))[:CALL_CAPACITY]
    return test.iloc[idx][column].value_counts(normalize=True).sort_index()

comparison = pd.DataFrame({
    "share of base":        test.handset_class.value_counts(normalize=True).sort_index(),
    "called, sees handset": called_mix(score_logit),
    "called, handset removed": called_mix(score_blind),
})
print(comparison.round(3).to_string())

overlap = len(set(np.argsort(-score_logit)[:CALL_CAPACITY]) &
              set(np.argsort(-score_blind)[:CALL_CAPACITY]))
print(f"\nSame people on both call lists: {overlap} of {CALL_CAPACITY} ({overlap/CALL_CAPACITY:.0%})")
print(f"precision@{CALL_CAPACITY}  with handset {precision_at_k(score_logit)[0]:.1%}"
      f"  |  without {precision_at_k(score_blind)[0]:.1%}")

**READ and answer.** The call list barely moved and precision barely moved.

> *A colleague proposes removing every demographic field from the model and calling
> the fairness problem solved. What is the one-sentence objection?*

This idea has a name, **fairness through unawareness**, and it is the most common
mistake in applied fairness work. You have removed the evidence of the targeting,
not the targeting.

---

# 3. Four definitions of fair

They are not shades of one idea. They are four different claims about what the
business owes, and they conflict.

In [ ]:
idx = np.argsort(-score_logit)[:CALL_CAPACITY]
audit_df = test.copy()
audit_df["score"]  = score_logit
audit_df["called"] = 0
audit_df.iloc[idx, audit_df.columns.get_loc("called")] = 1

GROUP = "handset_class"        # FILL: try "region" or "plan" instead

rows = []
for g, d in audit_df.groupby(GROUP):
    churners    = d[d.churned == 1]
    not_churners = d[d.churned == 0]
    rows.append({
        "group": g,
        "n": len(d),
        "base churn": d.churned.mean(),
        "share of base": len(d) / len(audit_df),
        "share of calls": d.called.sum() / CALL_CAPACITY,
        "TPR": churners.called.mean(),
        "FPR": not_churners.called.mean(),
        "precision if called": d[d.called == 1].churned.mean() if d.called.sum() else np.nan,
    })
fairness = pd.DataFrame(rows)
print(fairness.round(3).to_string(index=False))

In [ ]:
# The four tests, each reduced to the single number it turns on.
parity_gap = (fairness["share of calls"] - fairness["share of base"]).abs().max()
tpr_gap    = fairness.TPR.max() - fairness.TPR.min()
fpr_gap    = fairness.FPR.max() - fairness.FPR.min()

print(f"1. Demographic parity   largest gap between share of calls and share of base: {parity_gap:>6.1%}")
print(f"2. Equal opportunity    spread in true positive rate across groups:           {tpr_gap:>6.1%}")
print(f"3. Calibration          see the table below")
print(f"4. Error cost parity    not computable from data alone. It needs the business")
print(f"                        to say whether a missed churner costs the same in")
print(f"                        every group. That is a conversation, not a metric.")
print(f"\n(FPR spread, for reference: {fpr_gap:.1%})")

In [ ]:
# Calibration: does a given score mean the same thing for every group?
audit_df["band"] = pd.qcut(audit_df.score, 5, labels=["Q1 low","Q2","Q3","Q4","Q5 high"])
calibration = audit_df.pivot_table(index="band", columns=GROUP, values="churned",
                                   aggfunc="mean", observed=True)
print("Observed churn rate within each score band\n")
print(calibration.round(3).to_string())

top = calibration.loc["Q5 high"]
print(f"\nTop band spread: {top.min():.1%} to {top.max():.1%}")

## 3.1 You cannot have all four

Look at the two results together.

**Calibration holds.** In the top score band, observed churn is within a couple of
points for every group. A score of 0.3 means the same thing whoever receives it.

**Equal opportunity fails.** Among subscribers who actually churned, one group is
flagged materially more often than another.

Both are correct. This is not a bug you can fix.

In [ ]:
print("Base churn rate differs by group:")
for _, r in fairness.sort_values("base churn").iterrows():
    print(f"  {r.group:<14} {r['base churn']:.1%}")
print(f"\nSpread: {fairness['base churn'].min():.1%} to {fairness['base churn'].max():.1%}")
print("""
When base rates differ between groups, a calibrated score CANNOT produce equal error
rates. Kleinberg, Mullainathan and Raghavan (2016) and Chouldechova (2017) proved it.
It is arithmetic, not a modelling failure.

The consequence for you: fairness is not a box an analyst ticks. It is a definition
the business chooses, in writing, with a name against it.
""")

### BUILD: run the same audit on a different attribute

In [ ]:
# BUILD: set GROUP to "region" or "plan" above and re-run sections 3 and 3.1.
#
#   (a) Does the impossibility still bite? Check whether base rates differ.
#   (b) Which definition would you hold Safiri to for THAT attribute, and why
#       might it differ from your answer for handset class?

# YOUR CODE HERE

---

# 4. Parameters drift, and the inputs do not tell you

Notebook 05 §3 taught you PSI on the input features. This section shows you why that
is not enough.

In [ ]:
def population_stability_index(expected, actual, bins=10):
    """From NB05 §3. <0.1 stable, 0.1-0.25 moderate, >0.25 significant."""
    edges = np.quantile(expected, np.linspace(0, 1, bins + 1))
    edges[0], edges[-1] = -np.inf, np.inf
    e = np.histogram(expected, edges)[0] / len(expected) + 1e-6
    a = np.histogram(actual,   edges)[0] / len(actual)   + 1e-6
    return float(np.sum((a - e) * np.log(a / e)))

psi = pd.DataFrame([{"feature": c,
                     "PSI": population_stability_index(train[c].values, test[c].values)}
                    for c in FEATURES_NUM]).sort_values("PSI", ascending=False)
psi["verdict"] = pd.cut(psi.PSI, [-np.inf, .1, .25, np.inf],
                        labels=["stable", "moderate", "significant"])
print(psi.round(4).to_string(index=False))
print(f"\nEvery feature below the 0.25 retrain threshold. Maximum is {psi.PSI.max():.3f}.")
print("By input monitoring alone, this model is healthy.")

In [ ]:
# Now retrain the SAME specification on a later window and compare the parameters.
def coefficients_for(months):
    d = enrich(pd.concat([build_snapshot(panel, m) for m in months], ignore_index=True),
               False, THRESH)
    m = Pipeline([("pre", preprocessor()),
                  ("clf", LogisticRegression(max_iter=3000, class_weight="balanced", C=5))]
                ).fit(d[NUM+FLAG+CAT], d.churned)
    names = [f.split("__", 1)[1] for f in m.named_steps["pre"].get_feature_names_out()]
    return pd.Series(m.named_steps["clf"].coef_[0], index=names)

early, late = coefficients_for([3, 4]), coefficients_for([6, 7])
drift = pd.DataFrame({"months 3-4": early, "months 6-7": late})
drift["change"]    = drift["months 6-7"] - drift["months 3-4"]
drift["sign flip"] = np.where(np.sign(drift["months 3-4"]) != np.sign(drift["months 6-7"]), "YES", "")

print(drift.reindex(drift.change.abs().sort_values(ascending=False).index)
           .head(10).round(3).to_string())
print(f"\nCoefficients that flipped sign: {(drift['sign flip']=='YES').sum()} of {len(drift)}")

**READ and answer.** Put the two cells side by side.

Every input distribution says stable. Meanwhile `tenure_months` has collapsed toward
zero and several coefficients have changed sign outright.

> *What happened in month 6, and why did PSI not see it?*

This is the **concept drift** from NB05 §3, and here is the part that notebook could
not show you: the relationship changed while the inputs did not. Tenure stopped
protecting anybody when the competitor launched. The subscribers are the same, their
behaviour distributions are the same, and the model's understanding of them is wrong.

**Monitor your parameters, not only your inputs.** A coefficient sign flip between
retrains is a louder signal than any PSI threshold.

In [ ]:
# A monitor you can actually ship. Twenty lines.
def coefficient_stability_report(reference, current, floor=0.05, collapse_tol=0.5):
    """A sign flip only counts if at least one side is material. Without the floor
    the monitor fires on coefficients that were never doing anything."""
    out = []
    for name in reference.index:
        a, b = reference[name], current.get(name, np.nan)
        material  = max(abs(a), abs(b)) >= floor
        flipped   = np.sign(a) != np.sign(b)
        collapsed = abs(a) > 0.1 and abs(b) < abs(a) * collapse_tol
        alert = ("SIGN FLIP" if (flipped and material)
                 else "COLLAPSED" if collapsed
                 else "noise" if flipped else "")
        out.append({"coefficient": name, "was": a, "now": b, "alert": alert})
    return pd.DataFrame(out)

full   = coefficient_stability_report(early, late)
report = full[full.alert.isin(["SIGN FLIP","COLLAPSED"])].sort_values("alert")
noise  = full[full.alert == "noise"]

print(report.round(3).to_string(index=False))
print(f"\n{len(report)} coefficients need review before this retrain is approved.")
print(f"{(report.alert=='SIGN FLIP').sum()} material sign flips, "
      f"{(report.alert=='COLLAPSED').sum()} collapsed toward zero.")
if len(noise):
    print(f"\nSuppressed as noise: {', '.join(noise.coefficient)}")
    print("Those flipped sign too, but were never material. Without the floor your")
    print("monitor fires on them every retrain - the section 6 problem, one section early.")

---

# 5. The monitoring dashboard

Five tiles. Every number carries a comparator, because a number without one cannot
drive a decision. Every tile names the decision it informs, and a tile that cannot
is deleted.

In [ ]:
p_model, _ = precision_at_k(score_gb)
p_rule,  _ = precision_at_k(score_rule)

tiles = [
    ("Realised precision @500", f"{p_model:.1%}",
     f"vs break-even {BREAKEVEN:.1%} · vs rule {p_rule:.1%}", "Keep running the campaign?"),
    ("Net campaign value", f"KES {campaign_value(score_gb):,.0f}",
     f"vs rule KES {campaign_value(score_rule):,.0f}", "Which instrument to use"),
    ("Max feature PSI", f"{psi.PSI.max():.3f}",
     "vs 0.25 retrain threshold", "Retrain now?"),
    ("Material sign flips", f"{(report.alert=='SIGN FLIP').sum()} of {len(drift)}",
     "vs 0 expected", "Has the model changed?"),
    ("Scoring coverage", f"{len(test):,}",
     f"of {len(test):,} eligible", "Is the pipeline alive?"),
]
print(f"{'tile':<26}{'value':>14}  {'comparator':<38}{'informs'}")
print("-"*118)
for t, v, c, i in tiles:
    print(f"{t:<26}{v:>14}  {c:<38}{i}")

**Notice what is not on it.**

AUC. Nobody can act on an AUC. It does not tell you to call more people, retrain,
switch back to the rule or wake anybody up.

That is the test for every tile you will ever build: **if you cannot write the
"informs" column, the tile goes.**

---

# 6. The alert register

Notebook 05 §4 gave you retraining triggers. A trigger is not an alert. An alert has
a tier, an owner, an action and a review date, and if nobody acts on it, it is retired.

In [ ]:
register = pd.DataFrame([
    ["Feature contract breach", "Page", "ML engineer on call",
     "Stop scoring. Fix the feed before the next run", 2, 1.00],
    ["Scoring job failed", "Page", "ML engineer on call",
     "Rerun. If it fails twice, fall back to the complaints rule", 1, 1.00],
    ["Precision below break-even, 2 campaigns", "Page", "Head of Retention",
     "Switch the campaign back to the rule. Retrain", 0.3, 1.00],
    ["PSI above 0.25 on a top-five feature", "Digest", "Data science lead",
     "Investigate before the monthly retrain", 4, 0.40],
    ["Coefficient sign flip on retrain", "Digest", "Data science lead",
     "Review the model card. Re-run the fairness tests", 0.3, 0.90],
    ["Any feature PSI above 0.10", "KILL", "nobody",
     "Replace with the 0.25 rule above", 120, 0.02],
    ["Model scored N subscribers today", "Log", "Analytics",
     "No action. Kept for audit", 30, 0.00],
], columns=["alert","tier","owner","action","fires_per_month","acted_on"])

print(register[["alert","tier","owner","fires_per_month","acted_on"]].to_string(index=False))

reaching = register[register.tier.isin(["Page","Digest"])].fires_per_month.sum()
print(f"\nAlerts generated per month : {register.fires_per_month.sum():.0f}")
print(f"Alerts reaching a person   : {reaching:.0f}")
print(f"Removed by the ICU test    : {register[register.tier=='KILL'].fires_per_month.sum():.0f}")

**READ and answer.** Look at the row marked KILL.

> *It is the only one of the seven that resembles standard ML monitoring advice.
> Why does it have to go?*

120 fires a month, acted on 2%. That is the car alarm. It does not merely waste
attention, it **trains the team to ignore the alert channel**, which is how the three
Page-tier alerts stop working too.

The rule that keeps a register honest: **track the share acted on, per alert, every 90
days, and retire anything nobody acts on.**

In [ ]:
# BUILD: your own alert register.
#
# Take the model you built in Session 2 and write five alerts for it. For each one:
#   tier, owner, action, and an honest guess at how often it fires.
# Then apply the ICU test: would you still be reading this alert in three months?

my_register = pd.DataFrame([
    # ["...", "Page|Digest|Log", "...", "...", 0, 0.0],
], columns=["alert","tier","owner","action","fires_per_month","acted_on"])
print(my_register.to_string(index=False) if len(my_register) else "YOUR CODE HERE")

---

# 7. The deployment decision

Everything in this notebook comes down to one table.

In [ ]:
options = pd.DataFrame([
    ["Complaints rule", campaign_value(score_rule), "Fully. One sentence.",
     "The incumbent. Already live, costs nothing"],
    ["Logistic regression", campaign_value(score_logit), "Yes. Signed coefficients.",
     "Auditable, and loses to the rule it would replace"],
    ["Gradient boosting", campaign_value(score_gb), "Not directly.",
     "Best money. Cannot explain an individual decision"],
], columns=["option","value per month KES","explainable to a subscriber?","the catch"])
print(options.to_string(index=False))

gap = campaign_value(score_logit) - campaign_value(score_rule)
print(f"\nThe auditable model is worth KES {gap:,.0f} a month against doing nothing new.")

**There is no clean answer, and that is the finding.**

Under the Kenya Data Protection Act, a decision based solely on automated processing
engages the data subject's rights, and "the model scored you 0.81" is not an
explanation. That argument pushes toward the auditable model. The auditable model is
worth less than the rule it would replace.

Post-hoc explanation methods exist for the booster. An approximation of an explanation
is not the same thing as a coefficient you can state, and you should say so rather
than let the distinction quietly disappear.

**Your job is not to resolve this.** It is to put all three rows in front of the person
who can, with the trade named. Notebook 05 §5 has the governance detail. Confirm the
legal position with counsel or the Office of the Data Protection Commissioner.

---

## The handover checklist

Nothing goes live until every line has a name against it.

| | Where it came from |
|---|---|
| 1. The decision it serves, and who acts on the output | NB00 Part 1 |
| 2. The label, the prediction point, who agreed the definition | NB00 Parts 3-4 |
| 3. The non-ML baseline and the margin, in currency | NB00 Parts 8-9 |
| 4. The fairness definition chosen, and who signed it | §3 above |
| 5. Proxy test results for every attribute you care about | §2 above |
| 6. The feature contract, checked on every scoring batch | NB05 §2 |
| 7. The alert register: tier, owner, action, review date | §6 above |
| 8. The retraining trigger, and what runs while you retrain | §4 above, NB05 §4 |

Half of it you already had. Today finished it.